# Quality Control and Preprocessing

In [1]:
# Import modules

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as pyplot
import seaborn as sns
import scanpy as sc
import anndata as ad

In [2]:
# Load data

adata_uc = sc.read_h5ad("data/mmColon_single_cell_85K.h5ad")
print(adata_uc)

AnnData object with n_obs × n_vars = 84612 × 18416
    obs: 'sid', 'n_genes', 'condition', 'batch', 'sample', 'celltype_major', 'celltype_minor', 'celltype_subset'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells'
    uns: 'CCI', 'CCI_sample', 'Celltype_marker_DB', 'DEG', 'DEG_stat', 'GSA_down', 'GSA_up'
    obsm: 'HiCAT_result'


# QC check

To check the quality and the integrity of the dataset, a QC is performed. This step involves the investigation of:

1. UMIs per cell (The entry in X of anndata format)
    A. (Very) Low UMIs could mean ambient RNA is captured, or there are empty droplets. 
    B. (Too) High UMIs could mean 2 cells are captured in 1 droplet. These are called doublets. 
2. Genes detected per cell (Counting Non-zero entries per cell)
    A. Low could mean the cell is of poor-quality
    B. (Unusually) high could mean a doublet might be captured. 

UMIs and genes together show the complexity of the sample. If a sample has high UMIs but only few genes are detected could mean there is some kind of amplification bias or dominant behavior exhibited by the transcript. 

3. % Mitochondrial genes - During stress, cytoplasmic mRNA could leak but the mitochondrial transcript could still remain. Metabolically active cells can have higher % mito. For instance, if a cell is active, it needs energy, so mitochondrial fraction would be up in these cells. It is important to inspect the distribution to see which cells could be impacted by the threshold. 

4. % Ribosomal genes
5. % Haemoglobin genes


In [6]:
# Identify % mito, % ribo and % hemoglobin
adata_uc.var["mt"] = adata_uc.var_names.str.startswith("MT-")
# ribosomal genes
adata_uc.var["ribo"] = adata_uc.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata_uc.var["hb"] = adata_uc.var_names.str.contains("^HB[^(P)]")

# Calculate qc metrics
sc.pp.calculate_qc_metrics(adata_uc, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)


Although the scanpy tutorial mentioned that mito in mice starts with mt-, in this paper it is MT-. Maybe that is how it was processed. 

In [7]:
# Reinspect adata_uc again
print(adata_uc)

AnnData object with n_obs × n_vars = 84612 × 18416
    obs: 'sid', 'n_genes', 'condition', 'batch', 'sample', 'celltype_major', 'celltype_minor', 'celltype_subset', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'CCI', 'CCI_sample', 'Celltype_marker_DB', 'DEG', 'DEG_stat', 'GSA_down', 'GSA_up'
    obsm: 'HiCAT_result'


Now the adata_uc object has metrics added in obs and var

In [ ]:
# Summarize the metrics
print(type(adata_uc.obs))

# adata is a pandas data frame, so to pick the columns, use [[]]
adata_uc.obs[["n_genes_by_counts","total_counts","pct_counts_mt","pct_counts_ribo","pct_counts_hb"]].describe()

<class 'pandas.core.frame.DataFrame'>


,n_genes_by_counts,total_counts,pct_counts_mt,pct_counts_ribo,pct_counts_hb
count,84612.000000,84612.000000,84612.000000,84612.000000,84612.000000
mean,1931.992424,7409.290527,6.358524,19.912085,0.025477
std,1233.752926,8149.110352,10.198373,11.360708,0.834893
min,200.000000,407.000000,0.000000,0.299065,0.000000
25%,1108.000000,2958.000000,2.201895,10.499398,0.000000
50%,1593.000000,4886.500000,3.318221,18.410825,0.000000
75%,2615.000000,8988.250000,5.284570,29.144129,0.010400
max,9105.000000,167951.000000,95.278099,59.881424,82.859909


In the summary, minimum genes per cell is 200, it could mean that a minimum filter was applied by the authors (However, it is not mentioned in the paper, so it could also mean that in this experiment, there were no samples with detected genes below 200 --> brings to next exploration). Minimum gene counts is 407 which means there are cells with less than 500 genes. 

In [ ]:
# Per sample, investigate summary and write it in a csv



,condition
AAACCCACAGGATTCT-1-0-0-0-0-0-0-0-0-0,ChronicColitis
AAACCCACAGTCGCTG-1-0-0-0-0-0-0-0-0-0,ChronicColitis
AAACCCAGTGTCTTAG-1-0-0-0-0-0-0-0-0-0,ChronicColitis
AAACCCAGTTTACCTT-1-0-0-0-0-0-0-0-0-0,ChronicColitis
AAACCCATCCGTATGA-1-0-0-0-0-0-0-0-0-0,ChronicColitis
...,...
TTTGTTGAGCAATTCC-1-1,Healthy
TTTGTTGAGTGGCAGT-1-1,Healthy
TTTGTTGCACATGGTT-1-1,Healthy
TTTGTTGGTAACCAGG-1-1,Healthy


In [ ]:
# Visualize distribution per sample for each of the metrics with histograms

Look at valleys

In [ ]:
# Use scatter plots to visualize UMI vs genes per cell colored by mitochondrial percentage



In [ ]:
# % Mito - What happens if 15% threshold is applied


# Which cells are most affected?

Epithelial cells is expected, as this is colon tissue and dissociation of colon tissue could result in high % mito. But is it really technical or has some biology?

In [ ]:
# Low UMI and Low genes have to be removed - How to set threshold?

In [ ]:
# Doublet detection

In [ ]:
# Applying initial filtering thresholds and inspecting diagnostics

## Preprocessing

In [ ]:
# Normalization

In [ ]:
# Feature selection and Dimensionality reduction

In [ ]:
# Leiden clustering